# 05 — Full-scale QLoRA SFT on Kaggle T4

Max practical public-data run for **Llama-3.2-3B-Instruct** (4-bit QLoRA) on a single **T4**.

This is **not** the smoke path (`01` / `04`). It:

1. Streams capped public HF samples (earnings transcripts primary)
2. Builds grounded pairs → filtered → diversity-selected splits (design band **3k–6k**)
3. Trains **one full epoch** (or a step cap if you set one)
4. Optionally rebuilds the RAG corpus + hybrid metrics on the same chunks
5. **Publishes** the adapter + model card to **Hugging Face** and pushes **metadata** to **GitHub**

| Knob | Safe default | Full T4 run |
|------|--------------|-------------|
| `RUN_TRAIN` | `False` | `True` |
| `DOWNLOAD_HF` | `True` | `True` |
| `MAX_STEPS` | `None` (full epoch) | `None` |
| `PUBLISH_HF` | `False` | `True` (needs `HF_TOKEN`) |
| `PUSH_GITHUB` | `False` | `True` (needs `GITHUB_TOKEN`) |
| Transcript samples | `400` | `400` |

**Seed:** `3407`. **Adapter:** `outputs/adapters/llama32-3b-ecra-sft/`

**Session budget (typical T4):** data build 10–40 min; 3B QLoRA 1 epoch on ~3–6k rows often 1–4 h. Publish before the session dies.

Public data only. Metrics in the model card come from JSON only (TBD if missing).


## 0. Knobs

Edit this cell only. Keep `RUN_TRAIN=False` until data cells succeed and a T4 is attached.


In [ ]:
# --- User knobs ---
RUN_TRAIN = False              # True = full Unsloth QLoRA on GPU
MAX_STEPS = None               # None = train for num_train_epochs (1); or e.g. 200 smoke
DOWNLOAD_HF = True             # stream public HF samples (needs network; HF_TOKEN optional)

# Per-source caps (aligned with DEFAULT_MAX_PER_SOURCE in ingest.py)
MAX_SAMPLES_TRANSCRIPTS = 400  # primary grounding corpus
MAX_SAMPLES_FIQA = 200
MAX_SAMPLES_ALPACA = 150

CONFIG_PATH = "configs/default.yaml"  # 3B T4 path; 8B: configs/llama32-8b.yaml
DATASET_VERSION = "ecra-sft-v0.1.0"

# Optional post-train RAG on the same chunks
RUN_RAG = False
RUN_RAG_GENERATE = False       # needs adapter + GPU

# Post-train publish
PUBLISH_HF = False             # upload adapter + model card to Hugging Face
HF_REPO_ID = "nuwanda94/llama32-3b-ecra-sft"
PUSH_GITHUB = False            # commit sft_plan / manifests / metrics (not weights)
GITHUB_OWNER = "nuwanda94"
GITHUB_REPO = "earnings-call-research-assistant"
GITHUB_BRANCH = "main"

print("RUN_TRAIN", RUN_TRAIN, "MAX_STEPS", MAX_STEPS, "DOWNLOAD_HF", DOWNLOAD_HF)
print("caps:", MAX_SAMPLES_TRANSCRIPTS, MAX_SAMPLES_FIQA, MAX_SAMPLES_ALPACA)
print("RUN_RAG", RUN_RAG, "RUN_RAG_GENERATE", RUN_RAG_GENERATE)
print("PUBLISH_HF", PUBLISH_HF, HF_REPO_ID)
print("PUSH_GITHUB", PUSH_GITHUB)


## 1. Clone + path + installs


In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    if not (repo / "src" / "earnings_call_research_assistant" / "inference.py").exists():
        %cd /kaggle/working
        !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
        repo = work / "earnings-call-research-assistant"
    REPO = repo.resolve()
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)
print("cwd:", Path.cwd())

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__, "v", getattr(ecra, "__version__", "?"))


In [ ]:
if IN_KAGGLE:
    %pip install -q pyyaml rank_bm25
    %pip install -q datasets
    if RUN_TRAIN or RUN_RAG_GENERATE:
        %pip install -q unsloth transformers accelerate bitsandbytes trl peft
    if RUN_RAG:
        %pip install -q sentence-transformers
    if PUBLISH_HF:
        %pip install -q huggingface_hub
print("deps ready")


## 2. Secrets (HF + optional GitHub)

- `HF_TOKEN` — Hub stream + adapter upload (write scope for publish)
- `GITHUB_TOKEN` — classic PAT with `repo` scope for metadata push

Never print tokens.


In [ ]:
from earnings_call_research_assistant.data.ingest import resolve_hf_token, wire_hf_token

def _kaggle_secret(name: str):
    if not IN_KAGGLE:
        return None
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as e:
        print(f"No Kaggle secret {name}:", type(e).__name__)
        return None

tok = _kaggle_secret("HF_TOKEN")
if tok:
    wire_hf_token(tok)
    print("HF token wired: yes")
else:
    print("HF token wired:", "yes" if resolve_hf_token() else "no")

gh = _kaggle_secret("GITHUB_TOKEN")
if gh:
    os.environ["GITHUB_TOKEN"] = gh
    print("GITHUB_TOKEN wired: yes")
else:
    print("GITHUB_TOKEN wired:", "yes" if os.environ.get("GITHUB_TOKEN") else "no")


## 3. Ingest at T4-scale caps


In [ ]:
from earnings_call_research_assistant.data import (
    ingest_catalog,
    list_sources,
    write_jsonl,
)

print("Catalog:")
for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name}")

RAW = Path("data/raw/public_scale.jsonl")
records = ingest_catalog(
    source_ids=["earnings_transcripts", "fiqa", "finance_alpaca"],
    max_per_source={
        "earnings_transcripts": MAX_SAMPLES_TRANSCRIPTS,
        "fiqa": MAX_SAMPLES_FIQA,
        "finance_alpaca": MAX_SAMPLES_ALPACA,
    },
    download=DOWNLOAD_HF,
)
write_jsonl(records, RAW)
by = {}
for r in records:
    by[r.source_id] = by.get(r.source_id, 0) + 1
print(f"Wrote {len(records)} records -> {RAW.resolve()}")
print("by_source:", by)
assert len(records) > 0, "No records ingested"


## 4. Chunk + propositions


In [ ]:
from earnings_call_research_assistant.data import ChunkConfig, chunk_records, write_chunks_jsonl

CHUNKS = Path("data/processed/chunks.jsonl")
chunk_cfg = ChunkConfig(window_sentences=4, stride_sentences=2)
chunks = chunk_records(records, config=chunk_cfg)
write_chunks_jsonl(chunks, CHUNKS)
n_props = sum(len(c.propositions) for c in chunks)
print(f"chunks={len(chunks)} propositions={n_props} -> {CHUNKS}")
assert len(chunks) > 0


## 5. Grounded pairs (template, no LLM rewrite)


In [ ]:
from earnings_call_research_assistant.data import GenerateConfig, generate_pairs, write_pairs_jsonl

PAIRS = Path("data/processed/grounded_pairs.jsonl")
gen_cfg = GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False)
pairs = generate_pairs(chunks, config=gen_cfg)
write_pairs_jsonl(pairs, PAIRS)
n_qa = sum(1 for p in pairs if p.task == "qa")
n_sum = sum(1 for p in pairs if p.task == "summary")
print(f"pairs={len(pairs)} qa={n_qa} summary={n_sum} -> {PAIRS}")
assert len(pairs) > 0


## 6. Multi-stage filter


In [ ]:
from earnings_call_research_assistant.data import FilterConfig, filter_pairs, write_filter_report

FILTERED = Path("data/processed/filtered_pairs.jsonl")
REPORT = Path("data/processed/filter_report.json")
filt_cfg = FilterConfig(
    min_output_chars=40,
    near_dup_jaccard=0.88,
    use_llm_judge=False,
    min_judge_score=0.6,
)
kept, report = filter_pairs(pairs, config=filt_cfg)
write_pairs_jsonl(kept, FILTERED)
write_filter_report(report, REPORT)
print(
    f"in={report.n_in} kept={report.n_kept} "
    f"dropped={report.n_in - report.n_kept} by_stage={report.dropped_by_stage}"
)
assert report.n_kept > 0


## 7. Diversity select + versioned splits (3k–6k band)


In [ ]:
from earnings_call_research_assistant.data import (
    SelectConfig,
    select_and_split,
    write_splits,
)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=3000,
    target_max=6000,
    max_per_source=2500,
    max_per_task=4000,
    max_per_source_task=2000,
    diversity_jaccard_cap=0.72,
    seed=94,
    dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print(
    f"version={sel_report.dataset_version} selected={sel_report.n_selected} "
    f"train={sel_report.n_train} val={sel_report.n_val} test={sel_report.n_test}"
)
print("by_source=", sel_report.by_source)
print("by_task=", sel_report.by_task)
for k, v in paths.items():
    print(f"  {k}: {v}")
if sel_report.n_selected < 500:
    print(
        "WARNING: selected < 500 — HF stream may have failed or caps too low. "
        "Check by_source and re-run ingest with DOWNLOAD_HF=True + HF_TOKEN."
    )


## 8. SFT dry-run (always)


In [ ]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH,
    dataset_dir=OUT_DIR,
    dry_run=True,
    max_steps=MAX_STEPS,
    require_train=False,
)
print(
    f"dry_run={plan.dry_run} model={plan.model_name} seed={plan.seed} "
    f"train={plan.n_train} val={plan.n_val} effective_batch={plan.effective_batch_size}"
)
print("adapter_dir:", plan.adapter_dir)
print(json.dumps(plan.to_dict(), indent=2)[:2000])


## 9. Full QLoRA train on T4


In [ ]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

if not RUN_TRAIN:
    print("Skipped train (RUN_TRAIN=False). Set True and re-run this cell on T4.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("RUN_TRAIN=True but no CUDA. Enable GPU (T4) in Kaggle settings.")
    train_plan = run_sft(
        config_path=CONFIG_PATH,
        dataset_dir=OUT_DIR,
        dry_run=False,
        max_steps=MAX_STEPS,
        require_train=True,
    )
    print("Train finished.")
    print("adapter:", train_plan.adapter_dir)
    print("notes:", train_plan.notes[-5:])


## 10. Smoke-generate with adapter


In [ ]:
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness
import yaml

adapter = Path("outputs/adapters/llama32-3b-ecra-sft")
if not adapter.exists():
    print("Skip adapter smoke (no adapter on disk).")
else:
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    harness = InferenceHarness.from_pretrained(cfg)
    try:
        harness.model.load_adapter(str(adapter))
        print("Loaded adapter from", adapter)
    except Exception as e:
        print("Could not load_adapter (base-only smoke):", e)
    q = (
        "From a typical US large-cap earnings call, what is the difference "
        "between prepared remarks and the Q&A section for research?"
    )
    print(harness.generate(q))


## 11. Optional: scale RAG corpus + metrics


In [ ]:
import subprocess

if not RUN_RAG:
    print("Skipped RAG (RUN_RAG=False).")
else:
    steps = [
        [sys.executable, "scripts/build_rag_corpus.py", "--chunks", str(CHUNKS)],
        [sys.executable, "scripts/build_rag_index.py", "--run"],
        [sys.executable, "scripts/build_rag_eval_set.py", "--max-n", "50"],
        [sys.executable, "scripts/eval_retrieval.py", "--run"],
    ]
    for cmd in steps:
        print("==>", " ".join(cmd))
        rc = subprocess.call(cmd)
        if rc != 0:
            raise SystemExit(rc)
    if RUN_RAG_GENERATE and adapter.exists():
        cmd = [
            sys.executable,
            "scripts/eval_rag_generate.py",
            "--run",
            "--adapter-dir",
            str(adapter),
        ]
        print("==>", " ".join(cmd))
        rc = subprocess.call(cmd)
        if rc != 0:
            raise SystemExit(rc)
    man = Path("data/rag/corpus_v0.1.0/manifest.json")
    if man.is_file():
        snap = Path("evals/reports/corpus_manifest_snapshot.json")
        snap.write_text(man.read_text(encoding="utf-8"), encoding="utf-8")
        print("snapshot:", snap)
    print("RAG metrics under evals/reports/ — copy numbers only from JSON.")


## 12. Write model card + publish adapter to Hugging Face

Writes `outputs/adapters/llama32-3b-ecra-sft/README.md` from SFT plan + optional RAG JSON (TBD when missing), then uploads the folder.

Requires `PUBLISH_HF=True` and `HF_TOKEN`.


In [ ]:
import subprocess

adapter = Path("outputs/adapters/llama32-3b-ecra-sft")

if adapter.is_dir():
    rc = subprocess.call([sys.executable, "scripts/write_model_card.py", "--adapter-dir", str(adapter)])
    print("write_model_card rc:", rc)
else:
    print("No adapter dir yet; skip model card.")

if not PUBLISH_HF:
    print("Skipped HF upload (PUBLISH_HF=False). Dry-run plan:")
    subprocess.call([sys.executable, "scripts/publish_adapter.py", "--repo-id", HF_REPO_ID])
else:
    if not adapter.is_dir():
        raise FileNotFoundError(f"Adapter missing: {adapter}")
    if not resolve_hf_token():
        raise RuntimeError("PUBLISH_HF=True but no HF_TOKEN / HUGGING_FACE_HUB_TOKEN")
    cmd = [
        sys.executable,
        "scripts/publish_adapter.py",
        "--repo-id",
        HF_REPO_ID,
        "--run",
    ]
    print("==>", " ".join(cmd))
    rc = subprocess.call(cmd)
    if rc != 0:
        raise SystemExit(rc)
    print(f"Hub: https://huggingface.co/{HF_REPO_ID}")


## 13. Push run metadata to GitHub

Commits **small** artifacts only (`sft_plan.json`, dataset manifest, filter report, RAG metrics/report, PROGRESS/README). **Does not** push adapter weights or full JSONL corpora.

Requires `PUSH_GITHUB=True` and `GITHUB_TOKEN` (repo scope).


In [ ]:
cmd = [
    sys.executable,
    "scripts/push_run_metadata.py",
    "--owner", GITHUB_OWNER,
    "--repo", GITHUB_REPO,
    "--branch", GITHUB_BRANCH,
]
if PUSH_GITHUB:
    cmd.append("--run")
print("==>", " ".join(cmd))
rc = subprocess.call(cmd)
if rc != 0:
    raise SystemExit(rc)
if not PUSH_GITHUB:
    print("Dry-run only. Set PUSH_GITHUB=True after train to commit metadata.")


## Done

Checklist:

1. `data/processed/ecra-sft-v0.1.0/manifest.json` — train size
2. `outputs/sft_plan.json` — `seed: 3407`
3. Adapter at `outputs/adapters/llama32-3b-ecra-sft/` (+ `README.md` model card)
4. HF: https://huggingface.co/nuwanda94/llama32-3b-ecra-sft (if `PUBLISH_HF`)
5. GitHub: metrics/manifests committed (if `PUSH_GITHUB`)

CLI equivalents:

```bash
python scripts/write_model_card.py
python scripts/publish_adapter.py --repo-id nuwanda94/llama32-3b-ecra-sft --run
python scripts/push_run_metadata.py --run
```

See `docs/REPRODUCIBILITY.md` and `docs/MODEL_CARD_RAG.md`.
